[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaxiRuess/DeepLearning_101/blob/main/notebooks/06_Kernels/05_Fused_Attention.ipynb)

# Fused Attention — The Crown Jewel Kernel

This notebook implements a **FlashAttention-style fused attention kernel** in Triton. It computes $O = \text{softmax}(QK^T / \sqrt{d}) \cdot V$ **without ever creating the N x N attention matrix.**

This is the synthesis of every concept from the previous four notebooks:

| | LayerNorm (previous) | Fused Attention (this notebook) |
|---|---|---|
| Bottleneck | Memory-bound (HBM bandwidth) | **Memory-bound (N x N matrix in HBM)** |
| Grid | 1D (one program per row) | **1D (one program per query row)** |
| Key optimization | Fusion (1 pass vs 4+) | **Tiling + online softmax (avoid N x N)** |
| Triton advantage | Eliminates redundant HBM trips | **Eliminates the entire N x N matrix** |
| New concepts | Learnable params, two reductions | **K/V loop, online softmax state, rescaling** |

> We covered Flash Attention theory — memory hierarchy, online softmax, and Algorithm 1 — in the [Flash Attention notebook](../03_Training_Techniques/01_Flash_Attention.ipynb). That notebook explicitly noted that the pure-Python implementation can't be fast without a real kernel. **This notebook delivers that kernel.**

## Setup

In [6]:
import sys, os

if "google.colab" in str(get_ipython()):
    if not os.path.exists("/content/DeepLearning_101"):
        !git clone --depth 1 https://github.com/MaxiRuess/DeepLearning_101.git /content/DeepLearning_101
    os.chdir("/content/DeepLearning_101/notebooks/06_Kernels")
    sys.path.insert(0, "/content/DeepLearning_101")
else:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [7]:
import torch
import math

IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, '__IPYTHON__') else False
HAS_CUDA = torch.cuda.is_available()

if IN_COLAB:
    %pip install -q triton
    print(f"Running in Colab with GPU: {torch.cuda.get_device_name(0)}")
elif HAS_CUDA:
    print(f"Running locally with GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — will use Modal for remote GPU execution")
    print("Make sure you have Modal configured: pip install modal && modal token set")

No GPU detected — will use Modal for remote GPU execution
Make sure you have Modal configured: pip install modal && modal token set


## The Attention Memory Problem

Standard attention computes $S = QK^T / \sqrt{d}$, which is an **N x N matrix**:

| Sequence length N | Score matrix size | Q+K+V size (d=64) | Ratio |
|---|---|---|---|
| 512 | 1 MB | 384 KB | 2.7x |
| 1024 | 4 MB | 768 KB | 5.3x |
| 2048 | 16 MB | 1.5 MB | 10.7x |
| 4096 | 64 MB | 3 MB | 21.3x |

The score matrix grows **quadratically** while Q, K, V grow linearly. Worse, standard attention reads/writes this matrix through slow HBM **3 times**:

```
Naive attention (3 materializations of N x N):
  Step 1: S = Q @ K^T        → write N x N to HBM
  Step 2: P = softmax(S)     → read N x N, write N x N to HBM
  Step 3: O = P @ V          → read N x N from HBM

Fused attention (N x N never exists):
  For each query row q:
    Loop over K/V blocks (BLOCK_K rows, fit in SRAM):
      scores = q @ k_block^T        ← [BLOCK_K] vector in SRAM
      online softmax update          ← in SRAM
      accumulate weighted V          ← in SRAM
    Normalize + store one row of O
```

For the full theory on GPU memory hierarchy and IO complexity, see the [Flash Attention notebook](../03_Training_Techniques/01_Flash_Attention.ipynb).

## Online Softmax — The Key Trick

Standard softmax needs **all N scores** before it can produce any output. When we process K/V in blocks of BLOCK_K, we only see BLOCK_K scores at a time.

**Solution**: maintain three running state variables per query row:
- $m_i$ — running max of all scores seen so far
- $l_i$ — running sum of $e^{\text{scores} - m_i}$ (unnormalized denominator)
- $o_i$ — running weighted sum of V (unnormalized output)

When processing a new block with scores $s$:

$$m_{\text{new}} = \max(m_i, \max(s))$$
$$\alpha = e^{m_i - m_{\text{new}}} \quad \text{(rescale old state for new max)}$$
$$p = e^{s - m_{\text{new}}} \quad \text{(new block's weights)}$$
$$l_{\text{new}} = \alpha \cdot l_i + \sum p$$
$$o_{\text{new}} = \alpha \cdot o_i + p^T V_{\text{block}}$$

After all blocks: $O = o_i / l_i$

The **rescaling factor** $\alpha = e^{m_{\text{old}} - m_{\text{new}}}$ is the magic — when a new block has a larger max score, all previous accumulations are scaled down to maintain numerical stability.

In [8]:
def naive_attention(Q, K, V):
    """Standard attention — materializes the N x N score matrix."""
    d = Q.shape[-1]
    scores = Q @ K.T / math.sqrt(d)          # [N, N] <- the bottleneck
    attn_weights = torch.softmax(scores, dim=-1)  # [N, N]
    output = attn_weights @ V                 # [N, d]
    return output


# Memory comparison
for N in [512, 1024, 2048, 4096]:
    d = 64
    score_mb = N * N * 4 / 1024**2
    qkv_mb = 3 * N * d * 4 / 1024**2
    print(f"N={N:4d}: Score matrix = {score_mb:6.1f} MB, Q+K+V = {qkv_mb:.1f} MB, Ratio = {score_mb/qkv_mb:.1f}x")

N= 512: Score matrix =    1.0 MB, Q+K+V = 0.4 MB, Ratio = 2.7x
N=1024: Score matrix =    4.0 MB, Q+K+V = 0.8 MB, Ratio = 5.3x
N=2048: Score matrix =   16.0 MB, Q+K+V = 1.5 MB, Ratio = 10.7x
N=4096: Score matrix =   64.0 MB, Q+K+V = 3.0 MB, Ratio = 21.3x


## The Fused Attention Kernel

Our kernel processes **one query row per program**. For each query, it loops over all K/V rows in blocks of BLOCK_K, maintaining the online softmax state in SRAM. The N x N score matrix never exists — only BLOCK_K scores at a time.

This combines:
- **Tiling from [Matrix Multiply](./03_Matrix_Multiply.ipynb)** — loop over K/V in blocks
- **Online reduction from [Softmax](./02_Softmax.ipynb)** — running max/sum across tiles
- **Fusion from [Softmax](./02_Softmax.ipynb) + [LayerNorm](./04_LayerNorm.ipynb)** — entire attention in one kernel

The full code lives in `kernels/fused_attention.py`.

In [9]:
from kernels.fused_attention import fused_attention_kernel, fused_attention
import inspect
print(inspect.getsource(fused_attention_kernel))

ModuleNotFoundError: No module named 'triton'

## How the Kernel Works — Step by Step

```
Query row q (loaded once into SRAM):
  q = [q₀, q₁, ..., q_d]

K/V processed in blocks of BLOCK_K = 64:

  Block 0: k_start = 0
  ┌────────────────┐   ┌────────────────┐
  │ K[0:64, :]     │   │ V[0:64, :]     │
  │ (64 x D)       │   │ (64 x D)       │
  └────────────────┘   └────────────────┘
  scores = q @ K_block^T → [64]       (64 scores in SRAM)
  m_new = max(m_i, max(scores))        (update running max)
  alpha = exp(m_i - m_new)             (rescale factor)
  p = exp(scores - m_new)              (new weights)
  l_i = alpha * l_i + sum(p)           (update denominator)
  o_i = alpha * o_i + p^T @ V_block   (update output)
                    ↓
  Block 1: k_start = 64
  ┌────────────────┐   ┌────────────────┐
  │ K[64:128, :]   │   │ V[64:128, :]   │
  └────────────────┘   └────────────────┘
  (same update — Block 0's contribution is rescaled by alpha)
                    ↓
  ... (repeat for all blocks)
                    ↓
  Final: O[q_idx, :] = o_i / l_i      (normalize and store)
```

The largest intermediate at any point is:
- `q`: D floats (the query row)
- `k_block` + `v_block`: 2 x BLOCK_K x D floats (current K/V tile)
- `scores` + `p_ij`: 2 x BLOCK_K floats
- `o_i`: D floats (accumulator)

Total SRAM: ~2 x 64 x 64 x 4 = 32 KB. The N x N matrix (potentially MBs) never exists.

In [10]:
print(inspect.getsource(fused_attention))

NameError: name 'inspect' is not defined

## Concepts from Previous Kernels

| Concept | Source Notebook | How It Appears Here |
|---|---|---|
| Pointer arithmetic + masking | [01 Vector Add](./01_Vector_Add.ipynb) | `tl.load(K_ptr + offsets * stride, mask=...)` |
| Row-wise reduction (`tl.max`, `tl.sum`) | [02 Softmax](./02_Softmax.ipynb) | `tl.max(scores)`, `tl.sum(p_ij)` |
| Kernel fusion (avoid HBM round-trips) | [02 Softmax](./02_Softmax.ipynb) | Entire attention in one kernel, no N x N in HBM |
| Tiling + K-loop | [03 Matrix Multiply](./03_Matrix_Multiply.ipynb) | `for k_start in range(0, N, BLOCK_K)` |
| 2D pointer arithmetic (stride-based) | [03 Matrix Multiply](./03_Matrix_Multiply.ipynb) | `k_offsets[:, None] * stride_kn + d_offsets[None, :] * stride_kd` |
| Multiple external inputs | [04 LayerNorm](./04_LayerNorm.ipynb) | Q, K, V are separate tensors with their own strides |
| Online reduction across tiles | [02 Softmax](./02_Softmax.ipynb) + [Flash Attention](../03_Training_Techniques/01_Flash_Attention.ipynb) | `m_i`, `l_i`, `o_i` maintained across K/V blocks |

## Run on GPU

Triton requires an NVIDIA GPU. This notebook supports two execution modes:
- **Colab / Local CUDA** — runs directly on the available GPU
- **Modal** — runs on a remote T4 GPU (for Mac / no-GPU machines)

In [4]:
import time
import torch.nn.functional as F

def benchmark_attention():
    """Run correctness test + benchmark. Works on any CUDA device."""
    import triton
    import triton.language as tl

    @triton.jit
    def _fused_attention_kernel(
        Q_ptr, K_ptr, V_ptr, O_ptr,
        N, d,
        stride_qn, stride_qd, stride_kn, stride_kd,
        stride_vn, stride_vd, stride_on, stride_od,
        BLOCK_K: tl.constexpr, D: tl.constexpr,
    ):
        q_idx = tl.program_id(0)
        d_offsets = tl.arange(0, D)
        d_mask = d_offsets < d
        q = tl.load(Q_ptr + q_idx * stride_qn + d_offsets * stride_qd,
                    mask=d_mask, other=0.0).to(tl.float32)
        scale = 1.0 / tl.sqrt(tl.cast(d, tl.float32))
        m_i = float('-inf')
        l_i = 0.0
        o_i = tl.zeros((D,), dtype=tl.float32)
        for k_start in range(0, N, BLOCK_K):
            k_offsets = k_start + tl.arange(0, BLOCK_K)
            k_mask = k_offsets < N
            k_ptrs = K_ptr + k_offsets[:, None] * stride_kn + d_offsets[None, :] * stride_kd
            k_block = tl.load(k_ptrs, mask=k_mask[:, None] & d_mask[None, :], other=0.0).to(tl.float32)
            scores = tl.sum(q[None, :] * k_block, axis=1) * scale
            scores = tl.where(k_mask, scores, float('-inf'))
            m_ij = tl.max(scores, axis=0)
            m_new = tl.maximum(m_i, m_ij)
            alpha = tl.exp(m_i - m_new)
            p_ij = tl.exp(scores - m_new)
            l_new = alpha * l_i + tl.sum(p_ij, axis=0)
            v_ptrs = V_ptr + k_offsets[:, None] * stride_vn + d_offsets[None, :] * stride_vd
            v_block = tl.load(v_ptrs, mask=k_mask[:, None] & d_mask[None, :], other=0.0).to(tl.float32)
            o_i = alpha * o_i + tl.sum(p_ij[:, None] * v_block, axis=0)
            m_i = m_new
            l_i = l_new
        o_i = o_i / l_i
        o_ptrs = O_ptr + q_idx * stride_on + d_offsets * stride_od
        tl.store(o_ptrs, o_i, mask=d_mask)

    def triton_fused_attention(Q, K, V):
        N, d = Q.shape
        O = torch.empty_like(Q)
        BLOCK_K = 64
        D = triton.next_power_of_2(d)
        _fused_attention_kernel[(N,)](
            Q, K, V, O, N, d,
            Q.stride(0), Q.stride(1), K.stride(0), K.stride(1),
            V.stride(0), V.stride(1), O.stride(0), O.stride(1),
            BLOCK_K=BLOCK_K, D=D,
        )
        return O

    def naive_attention(Q, K, V):
        d = Q.shape[-1]
        scores = Q @ K.T / (d ** 0.5)
        return torch.softmax(scores, dim=-1) @ V

    # Correctness
    torch.manual_seed(0)
    N, d = 256, 64
    Q = torch.randn(N, d, device="cuda")
    K = torch.randn(N, d, device="cuda")
    V = torch.randn(N, d, device="cuda")
    out_triton = triton_fused_attention(Q, K, V)
    out_ref = naive_attention(Q, K, V)
    max_diff = (out_triton - out_ref).abs().max().item()
    match = torch.allclose(out_triton, out_ref, atol=1e-2)
    print(f"Max difference: {max_diff:.2e}, Results match: {match}")

    # Benchmark
    seq_lengths = [128, 256, 512, 1024, 2048]
    d = 64
    naive_times, sdpa_times, triton_times = [], [], []

    for N in seq_lengths:
        Q = torch.randn(N, d, device="cuda")
        K = torch.randn(N, d, device="cuda")
        V = torch.randn(N, d, device="cuda")

        for _ in range(10):
            naive_attention(Q, K, V)
            F.scaled_dot_product_attention(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0))
            triton_fused_attention(Q, K, V)
        torch.cuda.synchronize()

        for fn, times_list in [
            (lambda: naive_attention(Q, K, V), naive_times),
            (lambda: F.scaled_dot_product_attention(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)), sdpa_times),
            (lambda: triton_fused_attention(Q, K, V), triton_times),
        ]:
            start = time.perf_counter()
            for _ in range(100):
                fn()
            torch.cuda.synchronize()
            times_list.append((time.perf_counter() - start) / 100)

    return {
        "match": match, "max_diff": max_diff,
        "seq_lengths": seq_lengths,
        "naive_us": [t * 1e6 for t in naive_times],
        "sdpa_us": [t * 1e6 for t in sdpa_times],
        "triton_us": [t * 1e6 for t in triton_times],
    }

In [ ]:
if HAS_CUDA:
    results = benchmark_attention()
else:
    import modal

    app = modal.App("triton-fused-attention")
    image = modal.Image.debian_slim(python_version="3.12").pip_install("torch", "triton")

    @app.function(image=image, gpu="T4")
    def run_remote():
        import torch, torch.nn.functional as F, triton, triton.language as tl, time, math

        @triton.jit
        def _fused_attention_kernel(
            Q_ptr, K_ptr, V_ptr, O_ptr, N, d,
            stride_qn, stride_qd, stride_kn, stride_kd,
            stride_vn, stride_vd, stride_on, stride_od,
            BLOCK_K: tl.constexpr, D: tl.constexpr,
        ):
            q_idx = tl.program_id(0)
            d_offsets = tl.arange(0, D)
            d_mask = d_offsets < d
            q = tl.load(Q_ptr + q_idx * stride_qn + d_offsets * stride_qd,
                        mask=d_mask, other=0.0).to(tl.float32)
            scale = 1.0 / tl.sqrt(tl.cast(d, tl.float32))
            m_i = float('-inf')
            l_i = 0.0
            o_i = tl.zeros((D,), dtype=tl.float32)
            for k_start in range(0, N, BLOCK_K):
                k_offsets = k_start + tl.arange(0, BLOCK_K)
                k_mask = k_offsets < N
                k_ptrs = K_ptr + k_offsets[:, None] * stride_kn + d_offsets[None, :] * stride_kd
                k_block = tl.load(k_ptrs, mask=k_mask[:, None] & d_mask[None, :], other=0.0).to(tl.float32)
                scores = tl.sum(q[None, :] * k_block, axis=1) * scale
                scores = tl.where(k_mask, scores, float('-inf'))
                m_ij = tl.max(scores, axis=0)
                m_new = tl.maximum(m_i, m_ij)
                alpha = tl.exp(m_i - m_new)
                p_ij = tl.exp(scores - m_new)
                l_new = alpha * l_i + tl.sum(p_ij, axis=0)
                v_ptrs = V_ptr + k_offsets[:, None] * stride_vn + d_offsets[None, :] * stride_vd
                v_block = tl.load(v_ptrs, mask=k_mask[:, None] & d_mask[None, :], other=0.0).to(tl.float32)
                o_i = alpha * o_i + tl.sum(p_ij[:, None] * v_block, axis=0)
                m_i = m_new
                l_i = l_new
            o_i = o_i / l_i
            tl.store(O_ptr + q_idx * stride_on + d_offsets * stride_od, o_i, mask=d_mask)

        def triton_fused_attention(Q, K, V):
            N, d = Q.shape
            O = torch.empty_like(Q)
            BLOCK_K = 64
            D = triton.next_power_of_2(d)
            _fused_attention_kernel[(N,)](
                Q, K, V, O, N, d,
                Q.stride(0), Q.stride(1), K.stride(0), K.stride(1),
                V.stride(0), V.stride(1), O.stride(0), O.stride(1),
                BLOCK_K=BLOCK_K, D=D,
            )
            return O

        def naive_attention(Q, K, V):
            d = Q.shape[-1]
            return torch.softmax(Q @ K.T / (d ** 0.5), dim=-1) @ V

        torch.manual_seed(0)
        N, d = 256, 64
        Q = torch.randn(N, d, device="cuda")
        K = torch.randn(N, d, device="cuda")
        V = torch.randn(N, d, device="cuda")
        out_triton = triton_fused_attention(Q, K, V)
        out_ref = naive_attention(Q, K, V)
        max_diff = (out_triton - out_ref).abs().max().item()
        match = torch.allclose(out_triton, out_ref, atol=1e-2)
        print(f"Max diff: {max_diff:.2e}, match: {match}")

        seq_lengths = [128, 256, 512, 1024, 2048]
        d = 64
        naive_times, sdpa_times, triton_times = [], [], []
        for N in seq_lengths:
            Q = torch.randn(N, d, device="cuda")
            K = torch.randn(N, d, device="cuda")
            V = torch.randn(N, d, device="cuda")
            for _ in range(10):
                naive_attention(Q, K, V)
                F.scaled_dot_product_attention(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0))
                triton_fused_attention(Q, K, V)
            torch.cuda.synchronize()
            for fn, times_list in [
                (lambda: naive_attention(Q, K, V), naive_times),
                (lambda: F.scaled_dot_product_attention(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)), sdpa_times),
                (lambda: triton_fused_attention(Q, K, V), triton_times),
            ]:
                start = time.perf_counter()
                for _ in range(100): fn()
                torch.cuda.synchronize()
                times_list.append((time.perf_counter() - start) / 100)
        return {
            "match": match, "max_diff": max_diff, "seq_lengths": seq_lengths,
            "naive_us": [t * 1e6 for t in naive_times],
            "sdpa_us": [t * 1e6 for t in sdpa_times],
            "triton_us": [t * 1e6 for t in triton_times],
        }

    with app.run():
        results = run_remote.remote()

In [ ]:
print(f"Fused Attention — Correctness: {'PASS' if results['match'] else 'FAIL'}, Max diff: {results['max_diff']:.2e}")

## Benchmark Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seq_lengths = results["seq_lengths"]

# --- Plot 1: Execution time ---
ax = axes[0]
ax.plot(seq_lengths, results["naive_us"], "^-.", label="Naive (materializes NxN)", linewidth=2, color="#e74c3c")
ax.plot(seq_lengths, results["sdpa_us"], "s--", label="F.scaled_dot_product_attention", linewidth=2, color="#3498db")
ax.plot(seq_lengths, results["triton_us"], "o-", label="Triton Fused Attention", linewidth=2, color="#2ecc71")
ax.set_xscale("log", base=2)
ax.set_xlabel("Sequence length (N)")
ax.set_ylabel("Time (microseconds)")
ax.set_title("Fused Attention: Execution Time")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 2: Memory comparison ---
ax = axes[1]
d = 64
N_range = np.array(seq_lengths)
naive_mem = N_range ** 2 * 4 / 1024**2
fused_mem = N_range * d * 4 / 1024**2
ax.plot(N_range, naive_mem, "^-.", label="Naive: $O(N^2)$ score matrix", linewidth=2, color="#e74c3c")
ax.plot(N_range, fused_mem, "o-", label="Fused: $O(N \\cdot d)$ working memory", linewidth=2, color="#2ecc71")
ax.set_xlabel("Sequence length (N)")
ax.set_ylabel("Memory (MB)")
ax.set_title("Theoretical Memory: Score Matrix vs Working Set")
ax.legend()
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Connection to Production FlashAttention

Our kernel processes **one query row per program**. Production FlashAttention tiles both Q and K/V for better parallelism:

| | Our Kernel | FlashAttention-1 | FlashAttention-2 | FlashAttention-3 |
|---|---|---|---|---|
| Q processing | 1 row per program | Block of rows | Block of rows | Block of rows |
| Loop order | Outer: K/V blocks | Outer: K/V blocks | **Outer: Q blocks** | Outer: Q blocks |
| Online softmax | Yes | Yes | Yes | Yes |
| Parallelism | N programs | N/B_r programs | Better occupancy | Warp specialization |
| Hardware | Any GPU | Any GPU | Any GPU | **H100 only** |

In practice, use **`torch.nn.functional.scaled_dot_product_attention`** — it dispatches to FlashAttention-2 automatically on CUDA with fp16/bf16.

## What to Notice

1. **No N x N matrix is ever created** — the largest intermediate is `[BLOCK_K]` scores (64 floats), not `[N, N]` scores (millions of floats). Memory goes from $O(N^2)$ to $O(N)$.

2. **Online softmax makes incremental computation possible** — without the running max/sum trick, we'd need all N scores before computing softmax. With it, we process BLOCK_K scores at a time and get the exact same result.

3. **Every concept from previous notebooks appears here** — pointer arithmetic (01), row-wise reductions (02), kernel fusion (02, 04), tiling with K-loop (03), stride-based 2D loads (03). This kernel is the synthesis of the entire series.

4. **Our kernel is simplified for education** — one query row per program, no causal masking, no multi-head support, single-precision only. Production FlashAttention tiles both Q and K/V and supports causal masks.

5. **`F.scaled_dot_product_attention` is the practical choice** — it dispatches to optimized FlashAttention-2 on CUDA with fp16/bf16. Our kernel exists to understand the algorithm, not to replace it.

6. **The rescaling factor $\alpha = e^{m_{\text{old}} - m_{\text{new}}}$ is the magic** — when a new block has a larger max score, all previous accumulations are scaled down. This single multiplication makes the online algorithm numerically stable and exact.

## Resources

- [FlashAttention (Dao et al., 2022)](https://arxiv.org/abs/2205.14135) — The original paper
- [FlashAttention-2 (Dao, 2023)](https://arxiv.org/abs/2307.08691) — Better parallelism and work partitioning
- [Flash Attention Notebook](../03_Training_Techniques/01_Flash_Attention.ipynb) — Theory, Python implementation, and PyTorch SDPA benchmarks
- [Triton Fused Attention Tutorial](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html) — Official Triton tutorial
- [GPU MODE Lectures](https://github.com/gpu-mode/lectures) — Community GPU programming course